### Library imports

In [1]:
import os
import numpy as np
import pandas as pd
import torch
import pickle
from sklearn.ensemble import GradientBoostingRegressor
import utils
from model import MyModel
import warnings

warnings.filterwarnings(
    "ignore",
    message="Feature table contains NaN. 0 is used to fill these NaNs"
)

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())

torch: 2.3.0
cuda available: True


### GMEL config

In [2]:
YEAR = 2015
NUM_HIDDEN_LAYERS = 1
EMBEDDING_SIZE = 128
MULTITASK_WEIGHTS = (0.5, 0.25, 0.25)

# DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
DEVICE = "cpu"
device = torch.device(DEVICE)

print("Using device:", device)
print("Config:", YEAR, NUM_HIDDEN_LAYERS, EMBEDDING_SIZE, MULTITASK_WEIGHTS)

Using device: cpu
Config: 2015 1 128 (0.5, 0.25, 0.25)


### Load data, build graph

In [3]:
data = utils.load_dataset(year=YEAR)

g = utils.build_graph_from_matrix(
    data["ct_adjacency_withweight"],
    data["node_feats"].astype(np.float32),
    device=DEVICE
).to(device)

distm = data["distm"]  # distance/time matrix used in training

print("num_nodes:", data["num_nodes"])
print("node_feats shape:", data["node_feats"].shape)
print("distm shape:", distm.shape)
print("2000th instance from test data: ", data['test'][2000])

num_nodes: 2168
node_feats shape: (2168, 65)
distm shape: (2168, 2168)
2000th instance from test data:  [25 19 23]


### Get embeddings and load prediction model

In [4]:
def get_embeddings(year, L, E, W, g, device, save_if_computed=True):
    emb_path = (
        f"./embeddings/censustract_embeddings_year{year}_"
        f"layers{L}_emb{E}_multitask{W}.npz"
    )

    if os.path.exists(emb_path):
        z = np.load(emb_path)
        src_emb, dst_emb = z["arr_0"], z["arr_1"]
        print("Loaded embeddings from:", emb_path, src_emb.shape, dst_emb.shape)
        return src_emb, dst_emb

    ckpt_path = f"./models/model_state_layers{L}_emb{E}_multitask{W}.pth"

    if not os.path.exists(ckpt_path):
        raise FileNotFoundError(
            "Embeddings and checkpoint are both missing.\n"
            f"Missing embeddings: {emb_path}\n"
            f"Missing checkpoint: {ckpt_path}\n"
            "Run GMEL training first to generate them."
        )

    model = MyModel(
        g=g,
        num_nodes=g.number_of_nodes(),
        in_dim=g.ndata["attr"].shape[1],
        h_dim=E,
        num_hidden_layers=L,
        device=str(device),
        reg_param=0
    ).to(device)

    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()

    with torch.no_grad():
        src_emb = model(g).detach().cpu().numpy()
        dst_emb = model.forward2(g).detach().cpu().numpy()

    if save_if_computed:
        np.savez(emb_path, src_emb, dst_emb)
        print("Computed and saved embeddings to:", emb_path)
    else:
        print("Computed embeddings from checkpoint:", ckpt_path)

    print("Embeddings shape:", src_emb.shape, dst_emb.shape)
    return src_emb, dst_emb

In [5]:
def construct_feat(triplets, src_emb, dst_emb, scaled_distm):
    feat_src = src_emb[triplets[:, 0]]
    feat_dst = dst_emb[triplets[:, 1]]
    feat_dist = scaled_distm[triplets[:, 0], triplets[:, 1]].reshape(-1, 1)
    X = np.concatenate([feat_src, feat_dst, feat_dist], axis=1)
    y = triplets[:, 2]
    return X, y

def load_gbrt(
    year,
    num_hidden_layers,
    embedding_size,
    multitask_weights,
    data,
    distm,
    construct_feat,
):
    src_emb, dst_emb = get_embeddings(
    year,
    num_hidden_layers,
    embedding_size,
    multitask_weights,
    g,
    device
)

    scaled_distm = distm / distm.max() * np.max([src_emb.max(), dst_emb.max()])

    new_path = (
        f"./models/gbrt_year{year}_"
        f"layers{num_hidden_layers}_emb{embedding_size}_"
        f"multitask{multitask_weights}.pkl"
    )

    if os.path.exists(new_path):
        with open(new_path, "rb") as f:
            gbrt = pickle.load(f)

        print("Loaded existing GBRT:", new_path)

    else:
        X_train, y_train = construct_feat(data["train"], src_emb, dst_emb, scaled_distm)

        gbrt = GradientBoostingRegressor(
            max_depth=2,
            random_state=2019,
            n_estimators=100
        )
        gbrt.fit(X_train, y_train)

        with open(new_path, "wb") as f:
            pickle.dump(gbrt, f)

        print("Trained and saved new GBRT:", new_path)

    return gbrt, src_emb, dst_emb, scaled_distm

In [6]:
gbrt, src_emb, dst_emb, scaled_distm = load_gbrt(
    YEAR,
    NUM_HIDDEN_LAYERS,
    EMBEDDING_SIZE,
    MULTITASK_WEIGHTS,
    data,
    distm,
    construct_feat,
)

Loaded embeddings from: ./embeddings/censustract_embeddings_year2015_layers1_emb128_multitask(0.5, 0.25, 0.25).npz (2168, 128) (2168, 128)
Loaded existing GBRT: ./models/gbrt_year2015_layers1_emb128_multitask(0.5, 0.25, 0.25).pkl


### Make predictions

In [7]:
def make_features_from_pairs(pairs_nodeid, src_emb, dst_emb, scaled_distm):
    """
    pairs_nodeid: np.array shape (K,2) with node_id indices [src, dst]
    """
    pairs_nodeid = np.asarray(pairs_nodeid, dtype=int)
    feat_src = src_emb[pairs_nodeid[:, 0]]
    feat_dst = dst_emb[pairs_nodeid[:, 1]]
    feat_dist = scaled_distm[pairs_nodeid[:, 0], pairs_nodeid[:, 1]].reshape(-1, 1)
    X = np.concatenate([feat_src, feat_dst, feat_dist], axis=1)
    return X

def predict_pairs_nodeid(pairs_nodeid, gbrt, src_emb, dst_emb, scaled_distm):
    X = make_features_from_pairs(pairs_nodeid, src_emb, dst_emb, scaled_distm)
    y_hat = gbrt.predict(X)
    return y_hat

In [8]:
print("Example feature dim:", make_features_from_pairs([[25, 19]], src_emb, dst_emb, scaled_distm).shape)

# Example: pick any valid node ids
pairs = np.array([[25, 19]], dtype=int)

y_hat = predict_pairs_nodeid(pairs, gbrt, src_emb, dst_emb, scaled_distm)
print("Pred:", list(zip(pairs.tolist(), y_hat.tolist())))

Example feature dim: (1, 257)
Pred: [([25, 19], 29.82793650419287)]


In [9]:
data = utils.load_dataset(year=YEAR)

node_feats = data["node_feats"].copy()
ct_adj = data["ct_adjacency_withweight"]
distm = data["distm"]

In [10]:
src_node = 25
dst_node = 19

In [11]:
pair = np.array([[src_node, dst_node]], dtype=int)

In [26]:
modified_node_feats = node_feats.copy()

feature_idx = 44  # example: landarearatio_resarea after BoroCT2010 is removed
modified_node_feats[src_node, feature_idx] += 30.0

In [27]:
g_modified = utils.build_graph_from_matrix(
    ct_adj,
    modified_node_feats.astype(np.float32),
    device="cpu"
).to(device)

In [28]:
device = torch.device("cpu")

In [29]:
ckpt_path = f"./models/model_state_layers{NUM_HIDDEN_LAYERS}_emb{EMBEDDING_SIZE}_multitask{MULTITASK_WEIGHTS}.pth"

model = MyModel(
    g=g_modified,
    num_nodes=g_modified.number_of_nodes(),
    in_dim=modified_node_feats.shape[1],
    h_dim=EMBEDDING_SIZE,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    device=str(device),
    reg_param=0
).to(device)

ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["state_dict"])
model.eval();

In [30]:
with torch.no_grad():
    src_emb_modified = model(g_modified).detach().cpu().numpy()
    dst_emb_modified = model.forward2(g_modified).detach().cpu().numpy()

In [31]:
src_emb_modified

array([[0.        , 0.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.        , 0.        , 0.        , ..., 0.        , 2.764762  ,
        1.5452921 ],
       [0.        , 0.        , 0.        , ..., 0.        , 5.223386  ,
        3.176437  ],
       ...,
       [0.        , 0.        , 0.        , ..., 0.        , 3.1615365 ,
        2.8084555 ],
       [0.        , 0.        , 0.        , ..., 0.        , 1.9603847 ,
        2.5132372 ],
       [0.        , 0.        , 0.        , ..., 0.        , 0.44464016,
        1.1701825 ]], dtype=float32)

In [32]:
scaled_distm_modified = (
    distm / distm.max() * np.max([src_emb_modified.max(), dst_emb_modified.max()])
)

In [33]:
X_pair_modified = make_features_from_pairs(
    pair,
    src_emb_modified,
    dst_emb_modified,
    scaled_distm_modified
)

In [34]:
def make_features_from_pairs(pairs_nodeid, src_emb, dst_emb, scaled_distm):
    pairs_nodeid = np.asarray(pairs_nodeid, dtype=int)
    feat_src = src_emb[pairs_nodeid[:, 0]]
    feat_dst = dst_emb[pairs_nodeid[:, 1]]
    feat_dist = scaled_distm[pairs_nodeid[:, 0], pairs_nodeid[:, 1]].reshape(-1, 1)
    return np.concatenate([feat_src, feat_dst, feat_dist], axis=1)

In [35]:
y_hat_modified = gbrt.predict(X_pair_modified)

print(y_hat_modified)

[13.70105521]


In [36]:
X_pair_original = make_features_from_pairs(
    pair,
    src_emb,
    dst_emb,
    scaled_distm
)

y_hat_original = gbrt.predict(X_pair_original)

print("Original:", y_hat_original[0])
print("Modified:", y_hat_modified[0])
print("Difference:", y_hat_modified[0] - y_hat_original[0])

Original: 29.82793650419287
Modified: 13.701055211018035
Difference: -16.126881293174833
